# 08 — Feature Encoding

## Objective

Convert categorical features into numerical representations suitable for machine learning.

In [0]:
import pandas as pd

df = spark.table("default.titanic_processed").toPandas()

## 1. Handle Missing Embarked Values

Fill the two missing Embarked values using the mode.

In [0]:
embarked_mode = df["Embarked"].mode()[0]
df["Embarked"] = df["Embarked"].fillna(embarked_mode)

## 2. Encode Sex

Map `male` to 0 and `female` to 1.

In [0]:
df["Sex"] = df["Sex"].map({
    "male": 0,
    "female": 1
})

In [0]:
df = pd.get_dummies(df, columns=["Embarked"], dtype=int)

In [0]:
df[["Embarked_C", "Embarked_Q", "Embarked_S"]].head(10)

,Embarked_C,Embarked_Q,Embarked_S
0,0,0,1
1,1,0,0
2,0,0,1
3,0,0,1
4,0,0,1
5,0,1,0
6,0,0,1
7,0,0,1
8,0,0,1
9,1,0,0


In [0]:
df = pd.get_dummies(df, columns = ["Title"], dtype = int)

In [0]:
df[[
    "Title_Master",
    "Title_Miss",
    "Title_Mr",
    "Title_Mrs",
    "Title_Other"
]].head(10)

,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Other
0,0,0,1,0,0
1,0,0,0,1,0
2,0,1,0,0,0
3,0,0,0,1,0
4,0,0,1,0,0
5,0,0,1,0,0
6,0,0,1,0,0
7,1,0,0,0,0
8,0,0,0,1,0
9,0,0,0,1,0


In [0]:
df = pd.get_dummies(df, columns = ["AgeGroup"], dtype = int)

In [0]:
df[[
    "AgeGroup_Child",
    "AgeGroup_Teen",
    "AgeGroup_Adult",
    "AgeGroup_Senior"
]].head(10)

,AgeGroup_Child,AgeGroup_Teen,AgeGroup_Adult,AgeGroup_Senior
0,0,0,1,0
1,0,0,1,0
2,0,0,1,0
3,0,0,1,0
4,0,0,1,0
5,0,0,0,0
6,0,0,1,0
7,1,0,0,0
8,0,0,1,0
9,0,1,0,0


In [0]:
df = pd.get_dummies(df, columns = ["FamilySizeGroup"], dtype = int)


In [0]:
df[[
    "FamilySizeGroup_Alone",
    "FamilySizeGroup_Small",
    "FamilySizeGroup_Medium",
    "FamilySizeGroup_Large"
]].head(10)

,FamilySizeGroup_Alone,FamilySizeGroup_Small,FamilySizeGroup_Medium,FamilySizeGroup_Large
0,0,1,0,0
1,0,1,0,0
2,1,0,0,0
3,0,1,0,0
4,1,0,0,0
5,1,0,0,0
6,1,0,0,0
7,0,0,1,0
8,0,1,0,0
9,0,1,0,0


In [0]:
df["Cabin_Deck"] = df["Cabin"].str[0]

In [0]:
df["Cabin_Deck"] = df["Cabin_Deck"].fillna("Unknown")

In [0]:
df["Cabin_Deck"].value_counts()

Cabin_Deck
Unknown    687
C           59
B           47
D           33
E           32
A           15
F           13
G            4
T            1
Name: count, dtype: int64

In [0]:
df = pd.get_dummies(df, columns= ["Cabin_Deck"], dtype = int)

## 3. Define Model Features and Target

Exclude identifier/text columns that are not directly used as model features. `Survived` is the target.

In [0]:
X = df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin", "Survived"])
y = df["Survived"]

X.shape, y.shape

((891, 33), (891,))

## 4. Save the Encoded Dataset

Persist the encoded data as a Delta table for the train/test and modeling stages.

In [0]:
spark_df = spark.createDataFrame(df)
spark_df.write.mode("overwrite").saveAsTable("default.titanic_encoded")

In [0]:
spark_df.count(), len(spark_df.columns)

(891, 38)

## Conclusion

The categorical features have been encoded, the model feature matrix and target have been defined, and the encoded dataset has been saved as `default.titanic_encoded`.